In [2]:
import edgar as et, os, requests
from dotenv import load_dotenv
et.set_identity(os.getenv("EDGAR_IDENTITY"))

C:\Users\supah\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class XBRLIngestor:
    def __init__(self, ticker: str):
        self.ticker = ticker.upper()

        company = et.Company(self.ticker)
        self.cik = int(company.cik)

        self.headers = {
            "User-Agent": os.getenv("EDGAR_IDENTITY")
        }

    def get_company_facts(self):
        cik = str(self.cik).zfill(10)

        url = (
            "https://data.sec.gov/api/xbrl/"
            f"companyfacts/CIK{cik}.json"
        )

        response = requests.get(
            url,
            headers=self.headers,
            timeout=30
        )

        response.raise_for_status()

        return response.json()

    def get_concept(self, concept, unit="USD"):
        data = self.get_company_facts()

        gaap = data["facts"]["us-gaap"]

        if concept not in gaap:
            return []

        concept_data = gaap[concept]

        units = concept_data.get("units", {})

        return units.get(unit, [])

In [4]:
ingestor = XBRLIngestor("AAPL")
data = ingestor.get_company_facts()
print(data.keys())
print(data["entityName"])

dict_keys(['cik', 'entityName', 'facts'])
Apple Inc.


In [5]:
print(data["facts"].keys())
gaap = data["facts"]["us-gaap"]
list(gaap.keys())[:10]

dict_keys(['dei', 'us-gaap'])


['AccountsPayable',
 'AccountsPayableCurrent',
 'AccountsReceivableNetCurrent',
 'AccruedIncomeTaxesCurrent',
 'AccruedIncomeTaxesNoncurrent',
 'AccruedLiabilities',
 'AccruedLiabilitiesCurrent',
 'AccruedMarketingCostsCurrent',
 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment',
 'AccumulatedOtherComprehensiveIncomeLossAvailableForSaleSecuritiesAdjustmentNetOfTax']

In [7]:
rev = ingestor.get_concept("RevenueFromContractWithCustomerExcludingAssessedTax")
for row in rev[:10]:
    print(row)

{'start': '2016-09-25', 'end': '2017-09-30', 'val': 229234000000, 'accn': '0000320193-19-000119', 'fy': 2019, 'fp': 'FY', 'form': '10-K', 'filed': '2019-10-31', 'frame': 'CY2017'}
{'start': '2017-10-01', 'end': '2017-12-30', 'val': 88293000000, 'accn': '0000320193-19-000010', 'fy': 2019, 'fp': 'Q1', 'form': '10-Q', 'filed': '2019-01-30'}
{'start': '2017-10-01', 'end': '2017-12-30', 'val': 88293000000, 'accn': '0000320193-19-000119', 'fy': 2019, 'fp': 'FY', 'form': '10-K', 'filed': '2019-10-31', 'frame': 'CY2017Q4'}
{'start': '2017-10-01', 'end': '2018-03-31', 'val': 149430000000, 'accn': '0000320193-19-000066', 'fy': 2019, 'fp': 'Q2', 'form': '10-Q', 'filed': '2019-05-01'}
{'start': '2017-12-31', 'end': '2018-03-31', 'val': 61137000000, 'accn': '0000320193-19-000066', 'fy': 2019, 'fp': 'Q2', 'form': '10-Q', 'filed': '2019-05-01'}
{'start': '2017-12-31', 'end': '2018-03-31', 'val': 61137000000, 'accn': '0000320193-19-000119', 'fy': 2019, 'fp': 'FY', 'form': '10-K', 'filed': '2019-10-31'

In [8]:
data = ingestor.get_company_facts()

print(data["entityName"])
print(list(data["facts"]["us-gaap"].keys())[:50])

revenue = ingestor.get_concept(
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)

print(revenue[:10])

Apple Inc.
['AccountsPayable', 'AccountsPayableCurrent', 'AccountsReceivableNetCurrent', 'AccruedIncomeTaxesCurrent', 'AccruedIncomeTaxesNoncurrent', 'AccruedLiabilities', 'AccruedLiabilitiesCurrent', 'AccruedMarketingCostsCurrent', 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment', 'AccumulatedOtherComprehensiveIncomeLossAvailableForSaleSecuritiesAdjustmentNetOfTax', 'AccumulatedOtherComprehensiveIncomeLossCumulativeChangesInNetGainLossFromCashFlowHedgesEffectNetOfTax', 'AccumulatedOtherComprehensiveIncomeLossForeignCurrencyTranslationAdjustmentNetOfTax', 'AccumulatedOtherComprehensiveIncomeLossNetOfTax', 'AdjustmentsToAdditionalPaidInCapitalSharebasedCompensationRequisiteServicePeriodRecognitionValue', 'AdjustmentsToAdditionalPaidInCapitalTaxEffectFromShareBasedCompensation', 'AdvertisingExpense', 'AllocatedShareBasedCompensationExpense', 'AllowanceForDoubtfulAccountsReceivableCurrent', 'AmortizationOfIntangibleAssets', 'AntidilutiveSecuritiesExcludedFromComp